# 보고동사로 다시 읽기 ② 실제 AI API로 분석하기

① 노트북과 같은 입력·검사·저장·화면을 쓰고, 분석 엔진만 **Anthropic Claude API**(`engine_api.py`)로 바꿉니다.

- **전송 안내:** 문장 원문(두 논문 모두 CC BY 4.0 공개 논문)이 Anthropic API로 전송됩니다. 초안 주석 CSV는 보내지 않습니다.
- **비용:** 호출할 때마다 요금이 듭니다. 기본값은 시험 문장 2개만 보내는 소량 실행이고, 전체 실행은 `FULL_RUN=True`로 따로 켜야 합니다.
- **키:** 코랩 왼쪽의 🔑(보안 비밀)에 `ANTHROPIC_API_KEY`로 넣거나, 실행 중 입력창에 붙여 넣습니다. 키는 노트북·로그·결과 파일에 저장되지 않습니다.
- **실패 처리:** 응답 형식이 틀리면 정해진 횟수까지 다시 부르고, 끝내 실패한 문장은 '실패'로 기록합니다. 로컬 결과로 몰래 채우지 않습니다. 응답 원본은 모두 보관합니다.
- 결과는 **후보**이며, 초안 주석과의 비교는 정확도가 아니라 일치도입니다.

## 1. 설치

In [1]:
print("설치 칸 생략(시험 환경)")

설치 칸 생략(시험 환경)


## 2. 파일 준비

In [2]:
# 파일 준비: rv_colab.zip 하나를 올리거나, 파일을 하나씩 올려도 됩니다.
# ZIP은 rv_colab.zip 하나만 새 임시 폴더에 풀고, 필수 파일이 모두 있을 때만 현재 폴더에 덮어써 옮깁니다
# (이전에 올린 파일이 새 판본과 섞이지 않게). ZIP과 개별 파일을 섞어 올리면 ZIP 내용이 우선합니다.
import os, zipfile, shutil, tempfile, hashlib
NEED = ["rvtool.py", "engine_local.py", "engine_api.py", "viewer_template.html",
        "schemas/core_v1.3.json", "schemas/whole_paper_hits_v0.1.json",
        "inputs/paper_a_input.txt", "inputs/paper_a_structure.tsv", "inputs/paper_b_input.txt", "inputs/paper_b_structure.tsv"]
OPTIONAL = ["annotations/paper_a_draft_v1.3.csv", "annotations/paper_b_draft_v1.3.csv"]
ZIP_NAME = "rv_colab.zip"
def install_zip(zpath):
    tmp = tempfile.mkdtemp(prefix="rv_zip_")
    with zipfile.ZipFile(zpath) as f: f.extractall(tmp)
    base = os.path.join(tmp, "rv_colab") if os.path.isdir(os.path.join(tmp, "rv_colab")) else tmp
    lacking = [rel for rel in NEED if not os.path.exists(os.path.join(base, rel))]
    if lacking:   # 새 ZIP 자체에 필수 파일이 다 있어야 복사한다(이전 파일이 빈자리를 채우지 않게)
        raise FileNotFoundError(f"올린 ZIP에 필수 파일이 없습니다: {lacking}. 아무것도 복사하지 않았습니다. 최신 {ZIP_NAME}을 올려 주세요.")
    for rel in NEED + OPTIONAL:
        src = os.path.join(base, rel)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(rel) or ".", exist_ok=True); shutil.copy2(src, rel)
    print(f"자료 ZIP: {zpath} (SHA-256 {hashlib.sha256(open(zpath, 'rb').read()).hexdigest()[:16]}…)")
def place_loose_files():
    # 하나씩 올린 자료 파일은 이름을 보고 제자리로 옮긴다(같은 이름이 있으면 새 파일로 바꾼다)
    for rel in NEED + OPTIONAL:
        name = os.path.basename(rel)
        if name != rel and os.path.exists(name):
            os.makedirs(os.path.dirname(rel), exist_ok=True); shutil.move(name, rel)
if os.path.exists(ZIP_NAME): install_zip(ZIP_NAME)
place_loose_files()
missing = [p for p in NEED if not os.path.exists(p)]
if missing:
    try:
        from google.colab import files
        print("다음 파일이 없습니다. 올려 주세요(rv_colab.zip 하나 또는 개별 파일):", missing)
        up = list(files.upload())
        zips = [n for n in up if n.endswith(".zip")]
        if len(zips) > 1: raise FileNotFoundError(f"ZIP 파일은 하나만 올려 주세요(올린 ZIP: {zips})")
        if zips: install_zip(zips[0])
        place_loose_files()
    except ImportError:
        pass
missing = [p for p in NEED if not os.path.exists(p)]
print("필수 파일:", "모두 있음" if not missing else f"없음 → {missing}")
print("선택 파일(초안 주석 CSV):", [p for p in OPTIONAL if os.path.exists(p)])
if missing: raise FileNotFoundError(f"필수 파일이 없습니다: {missing}. 올린 뒤 이 칸을 다시 실행하세요.")


필수 파일: 모두 있음
선택 파일(초안 주석 CSV): ['annotations/paper_a_draft_v1.3.csv', 'annotations/paper_b_draft_v1.3.csv']


## 3. 입력 검사

In [3]:
import sys, json, importlib
sys.path.insert(0, ".")
import rvtool as rv; importlib.reload(rv)
if not hasattr(rv, "save_validated_run"):
    raise RuntimeError(f"공통 모듈 판본이 이 노트북보다 오래되었습니다({rv.TOOL_VERSION}). 이 노트북과 같은 때 받은 rv_colab.zip을 올려 주세요.")
print("공통 모듈:", rv.TOOL_VERSION)
RUN_ROOT = "rv_runs"            # 실행 결과 저장 폴더(실행마다 새 하위 폴더, 덮어쓰지 않음)
DOC_VERSION = "preproc-v1.2-xml-20261015"   # 입력 판본 이름(입력 파일 머리부에 없어서 여기서 정함)
def load(d):
    ann = f"annotations/{d}_draft_v1.3.csv"
    return rv.load_document(f"inputs/{d}_input.txt", f"inputs/{d}_structure.tsv", d,
                            ann if os.path.exists(ann) else None, document_version=DOC_VERSION)
A, B = load("paper_a"), load("paper_b")
for D in (A, B): print(json.dumps(rv.describe(D), ensure_ascii=False, indent=1))


공통 모듈: rvtool-1.3.1
{
 "document_id": "paper_a",
 "document_version": "preproc-v1.2-xml-20261015",
 "sentences": 249,
 "paragraphs": 64,
 "by_section": {
  "ABS": 10,
  "INT": 24,
  "MET": 67,
  "RD": 139,
  "CON": 9
 },
 "citing_sentences": 49,
 "annotations_loaded": true,
 "hashes": {
  "txt": "9c5be835900932e2fe268efbaa8ba739021250ac60478718eac35370239f9c78",
  "tsv": "dde09f54fbb92d0bfc07384da94f731eba8c67889b4a248dad535e32048f84b9",
  "annotations": "fc53e53b22d18959859248ee4c9339fa5f27e3761616d57a80815e7ddb6fa873"
 }
}
{
 "document_id": "paper_b",
 "document_version": "preproc-v1.2-xml-20261015",
 "sentences": 215,
 "paragraphs": 48,
 "by_section": {
  "ABS": 8,
  "INT": 43,
  "RD": 73,
  "MET": 85,
  "CON": 6
 },
 "citing_sentences": 49,
 "annotations_loaded": true,
 "hashes": {
  "txt": "3a45b159f4e988a85747b9189d42801bbf394b456d966cec86763b0814e035a9",
  "tsv": "6a2bdf82816fc464447eeb4bad522198f54b66e16a789ff66daed574a6fa5848",
  "annotations": "ec5ec4daafb2f9c7edefae6c5f5ba7e

## 4. API 설정
모델 ID는 Anthropic 문서의 모델 목록(https://docs.claude.com/en/docs/about-claude/models)에서 확인해 넣으세요.

In [4]:
# [시험용 대체] 실제 API 대신 가짜 호출기. 네트워크 호출 없음. 두 번째 호출은 일부러 형식 오류를 냅니다.
import engine_api as API, json as _j
class FakeCaller:
    model="FAKE-no-network"; calls=0
    def settings(self): return dict(provider="fake", model=self.model)
    def __call__(self, system, user):
        FakeCaller.calls+=1
        p=_j.loads(user); D={"paper_a":A,"paper_b":B}[p["document_id"]]; m=D.by_id(); out=[]
        if FakeCaller.calls==2: return "형식이 틀린 응답", dict(usage={})
        for x in p["sentences"]:
            i=x["sentence_id"]; s=m[i].text; nums=rv.citation_numbers(s); a=D.annotations[i]
            v=a.get("reporting_verb","") if nums else ""
            out.append(dict(sentence_id=i,has_citation=bool(nums),citation_numbers=nums,reporting_verb=v,citation_form=a.get("citation_form","") if nums else "",note="",
                            hits=[dict(text=t,occurrence=1,role="prior",attr="선행연구",agent="선행연구") for t in v.split(", ") if t]))
        return _j.dumps(dict(results=out),ensure_ascii=False), dict(usage=dict(input_tokens=10,output_tokens=10))
CHUNK_SIZE, MAX_RETRIES = 20, 3
caller=FakeCaller(); print("가짜 호출기 사용:", caller.settings())

가짜 호출기 사용: {'provider': 'fake', 'model': 'FAKE-no-network'}


## 5. 소량 시험(기본)
시험 문장 2개만 보냅니다. **`CONFIRM_SEND`는 기본값이 `False`라서, 직접 `True`로 바꾸기 전에는 아무것도 전송되지 않습니다.** 성공하면 전체 실행과 똑같이 정식 저장 → 다시 읽기 → 검증 → 부분 화면까지 확인합니다. 이 결과의 성공은 연결이 작동한다는 뜻이지 분석 정확도의 검증이 아닙니다.

In [5]:
import time
CONFIRM_SEND = False   # 문장 원문을 API로 보내는 데 동의하면 직접 True로 바꾸세요(기본값 False)
ids = [i for d, i in rv.SMOKE]
print("예상:", API.estimate(A, ids, CHUNK_SIZE))
if not CONFIRM_SEND:
    raise SystemExit("전송하지 않았습니다. CONFIRM_SEND = True로 바꾼 뒤 이 칸을 다시 실행하세요.")
core, whole, f, ex = API.analyze(A, caller, RUN_ROOT, sentence_ids=ids, chunk_size=CHUNK_SIZE, max_retries=MAX_RETRIES)
ok_ids = [i for i in ids if i not in set(f["processing"]["failed_ids"])]
rec = rv.base_record(A, "api", f["run_id"], ["core_v1.3", "whole_paper_hits_0.1"], run_kind="smoke", **{k: v for k, v in f.items() if k != "run_id"})
SMOKE_RUN = f["run_id"]
SMOKE_DIR = rv.save_validated_run(RUN_ROOT, A, SMOKE_RUN, rec, core, whole, extra=ex, expected_ids=ok_ids)   # 정식 저장(검증 실패 시 저장하지 않음)
r = rv.load_run(RUN_ROOT, "paper_a", SMOKE_RUN)                                    # 다시 읽기(해시 확인)
for c in r["core"]: print(c)
print("표현:", [(h["sentence_id"], h["text"], h["role"]) for h in r["whole"]])
print("처리:", r["record"]["processing"], "| 사용량:", r["record"]["usage_total"])
print("다시 읽은 결과 검사 — core:", rv.validate_core(r["core"], A, ok_ids) or "통과", "| whole:", rv.validate_whole(r["whole"], A) or "통과",
      "| 위치를 찾지 못한 표현:", len(ex["invalid_hits"]))
SMOKE_VIEW = rv.build_viewer([r], [A], f"viewer_api_smoke_{time.strftime('%Y%m%d-%H%M%S')}.html")
print("저장:", SMOKE_DIR, "| 부분 화면:", SMOKE_VIEW)
try:
    from google.colab import files
    from IPython.display import HTML, display
    display(HTML(open(SMOKE_VIEW, encoding="utf-8").read()))
except ImportError:
    pass

예상: {'sentences': 2, 'chunks': 1, 'sentence_chars': 344, 'rough_input_tokens': 724, 'note': '대략적인 추정이며 실제 요금은 제공자 요금표로 확인하세요'}


SystemExit: 전송하지 않았습니다. CONFIRM_SEND = True로 바꾼 뒤 이 칸을 다시 실행하세요.

/private/tmp/claude-501/-Users-ykpark--claude-spares-spare-uzdzr9/7076eed7-3be6-4dab-b521-d1bc305f5f52/scratchpad/venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 6. 전체 실행(선택)
`FULL_RUN=True`이고 `CONFIRM_SEND=True`일 때만 실행됩니다. 실행을 시작하면 논문마다 `새 실행 ID`가 출력되니 적어 두세요. 중간에 끊기면 `RESUME={'paper_a': '실행ID', 'paper_b': '실행ID'}`에 **두 논문의 실행 ID를 모두** 넣고 다시 실행하세요. 이미 저장까지 끝난 논문은 다시 부르지 않고 저장된 결과를 그대로 쓰며, 끊긴 논문만 이어서 실행합니다. `RESUME`에 넣은 ID를 찾지 못하면(오타 등) 새 실행을 시작하지 않고 멈춥니다. 재개할 때는 입력 파일 해시·문장 목록·모델·max_tokens·묶음 크기·프롬프트가 처음 실행과 같은지 먼저 확인하고, 하나라도 다르면 멈춥니다. 끝난 묶음의 캐시도 다시 검증합니다.

In [ ]:
FULL_RUN = False
RESUME = {}          # 예: {"paper_a": "20261019-101500-api-1a2b3c"}
RUNS = {}
if FULL_RUN and not CONFIRM_SEND:
    raise SystemExit("CONFIRM_SEND가 False라 전송하지 않았습니다.")
if not FULL_RUN:
    print("전체 실행을 건너뜁니다(FULL_RUN=False). 예상:", {D.document_id: API.estimate(D, [s.id for s in D.sentences], CHUNK_SIZE) for D in (A, B)})
else:
    for D in (A, B):
        if API.is_saved(RUN_ROOT, D, RESUME.get(D.document_id)):   # 이미 저장까지 끝난 논문은 다시 부르지 않는다
            RUNS[D.document_id] = RESUME[D.document_id]
            print(D.document_id, "이미 저장이 끝난 실행이라 다시 부르지 않습니다:", RESUME[D.document_id]); continue
        core, whole, f, ex = API.analyze(D, caller, RUN_ROOT, run_id=RESUME.get(D.document_id), chunk_size=CHUNK_SIZE, max_retries=MAX_RETRIES)
        ok_ids = [s.id for s in D.sentences if s.id not in set(f["processing"]["failed_ids"])]
        rec = rv.base_record(D, "api", f["run_id"], ["core_v1.3", "whole_paper_hits_0.1"], **{k: v for k, v in f.items() if k != "run_id"})
        d = rv.save_validated_run(RUN_ROOT, D, f["run_id"], rec, core, whole, extra=ex, expected_ids=ok_ids)   # 검증 실패 시 저장하지 않음
        RUNS[D.document_id] = f["run_id"]
        print(D.document_id, "저장:", d, "| 처리:", f["processing"]["succeeded"], "성공 /", f["processing"]["failed"], "실패 | 사용량:", f["usage_total"])

## 7. 다시 읽기·일치도·집계
저장 후 해시를 확인하고 다시 검사합니다. 초안 주석과의 비교는 일치도입니다.

In [ ]:
for D in (A, B):
    if D.document_id not in RUNS: print(D.document_id, "저장된 API 실행 없음"); continue
    r = rv.load_run(RUN_ROOT, D.document_id, RUNS[D.document_id])
    failed = r["record"]["processing"]["failed_ids"]
    print(D.document_id, "| core:", rv.validate_core(r["core"], D, [s.id for s in D.sentences if s.id not in set(failed)]) or "통과",
          "| whole:", rv.validate_whole(r["whole"], D) or "통과", "| 실패 문장:", len(failed))
    ag = rv.agreement_with_annotations(r["core"], D, "독립 분석(API)")
    if ag: print("  ", ag["label"], ag["counts"], "불일치 후보", len(ag["disagreement_candidates"]))

## 8. 화면 연결
API 실행 결과로 화면을 만듭니다. 화면 맨 위에 어떤 엔진·실행·모델로 만든 결과인지 표시됩니다.

In [ ]:
import time
if len(RUNS) == 2:
    out = f"viewer_api_{time.strftime('%Y%m%d-%H%M%S')}.html"
    rv.build_viewer([rv.load_run(RUN_ROOT, "paper_a", RUNS["paper_a"]), rv.load_run(RUN_ROOT, "paper_b", RUNS["paper_b"])], [A, B], out)
    print("화면 파일:", out)
    try:
        from google.colab import files
        from IPython.display import HTML, display
        display(HTML(open(out, encoding="utf-8").read())); files.download(out)
    except ImportError:
        pass
else:
    print("두 논문의 전체 API 실행이 있어야 화면을 만듭니다.")

## 9. 결과 내려받기

In [ ]:
import shutil
z = shutil.make_archive(f"rv_runs_api_{time.strftime('%Y%m%d-%H%M%S')}", "zip", RUN_ROOT)
print("내려받을 파일:", z)
try:
    from google.colab import files; files.download(z)
except ImportError:
    pass